# Self-sizing TabM + distillation: POST-HOC, EXPLORATORY

Standalone and resumable. Run all cells in order on a GPU runtime (Colab T4 is enough).

**Question.** Does distilling the default (full-size) TabM into the self-sizing student recover the accuracy
that the frozen shrink-during-training schedule loses on cpu_act and superconduct?

**Status.** These two splits are where the frozen method *lost*, so choosing this fix after seeing them makes the
result a *diagnostic*, not a validated improvement. Do not put it in the frozen test table. It needs a tuning
check and a fresh test in the next round.

**Pre-declared protocol (nothing is tuned here).**
- Splits: cpu_act medium-0 and superconduct medium-0. Seeds 0, 1, 2.
- Student = the frozen self-sizing setting, unchanged (`FROZEN_SS`). The only difference is the training target.
- Teacher = default TabM, same seed and split (its saved ensemble-mean prediction on the training rows).
- Two arms, both fixed in advance: lambda = 0.5 and lambda = 1.0. Both are reported; neither is picked after the fact.
- Compared against: default TabM, frozen self-sizing, and the plain model at the frozen width.

**Method (exact).** For regression, `(1-l)(p-y)^2 + l(p-t)^2 = (p-m)^2 + l(1-l)(y-t)^2` with `m = (1-l)y + l t`.
The last term does not depend on the prediction `p`, so the two-term loss and a single MSE against the blended
target `m` have identical gradients. The patch therefore only replaces `Y_train` by `m` once, before training.
Everything else (gates, penalty, pruning, early stopping) is untouched, and with `distill.enabled = false` the new
script is line-for-line the frozen one (the patch only adds code).

**Known weakness.** The teacher has fit the training rows, so on them it is close to the labels and `m` is close to
`y`. The summary prints the teacher's train vs validation error. If the teacher's train error is tiny, a null result
is expected and says nothing against distillation in general.

## Cell 1: setup (same as the earlier notebooks, repo pinned)

In [ ]:
import os, subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tabm_selfsizing'          # same folder as the Step 3/4 notebook
os.makedirs(ROOT, exist_ok=True)

COMMIT = '28e47ae301c92ec37787dde1ce923a0793f405b4'      # the repo version this code was tested against
if not os.path.exists('/content/tabm'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yandex-research/tabm', '/content/tabm'], check=True)
    subprocess.run(['git', '-C', '/content/tabm', 'checkout', '-q', COMMIT], check=True)
PAPER = '/content/tabm/paper'
os.chdir(PAPER)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'delu==0.0.25', 'rtdl_num_embeddings==0.0.11', 'rtdl_revisiting_models==0.0.2'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'loguru', 'tomli', 'tomli-w', 'optuna'], check=True)

util = open('lib/util.py').read()
old = 'return _torch().load(get_checkpoint_path(output), **kwargs)'
new = 'return _torch().load(get_checkpoint_path(output), **{"weights_only": False, **kwargs})'
if old in util:
    open('lib/util.py', 'w').write(util.replace(old, new))
print('Patched:', new in open('lib/util.py').read())

name, memory.total [MiB]
Tesla T4, 15360 MiB

Mounted at /content/drive
Patched: True


## Cell 2: data and dataset lists

In [ ]:
import glob
DATA_DRIVE = f'{ROOT}/data'
if not os.path.exists(f'{DATA_DRIVE}/california'):
    os.makedirs(f'{ROOT}/local', exist_ok=True)
    subprocess.run(['wget', '-q', 'https://huggingface.co/datasets/rototoHF/tabm-data/resolve/main/data.tar',
                    '-O', f'{ROOT}/local/tabm-data.tar.gz'], check=True)
    os.makedirs(DATA_DRIVE, exist_ok=True)
    subprocess.run(['tar', '-xf', f'{ROOT}/local/tabm-data.tar.gz', '-C', DATA_DRIVE], check=True)
    inner = [p for p in glob.glob(f'{DATA_DRIVE}/*') if os.path.isdir(p)]
    if len(inner) == 1 and not os.path.exists(f'{inner[0]}/info.json'):
        subprocess.run(f'mv {inner[0]}/* {DATA_DRIVE}/ && rmdir {inner[0]}', shell=True, check=True)
if not os.path.islink(f'{PAPER}/data'):
    if os.path.exists(f'{PAPER}/data'): subprocess.run(['rm', '-rf', f'{PAPER}/data'])
    os.symlink(DATA_DRIVE, f'{PAPER}/data')
os.makedirs(f'{ROOT}/exp_runs', exist_ok=True)
if not os.path.islink(f'{PAPER}/exp/runs'):
    os.symlink(f'{ROOT}/exp_runs', f'{PAPER}/exp/runs')

# The 12 TEST datasets: never used to choose settings
DATASETS = {
    'california': ['california'], 'house': ['house'], 'diamond': ['diamond'],
    'cpu_act': [f'why/regression-num-medium-{i}-cpu_act' for i in range(3)],
    'superconduct': ['why/regression-num-medium-0-superconduct'],
    'churn': ['churn'], 'adult': ['adult'], 'higgs-small': ['higgs-small'],
    'MagicTelescope': [f'why/classif-num-medium-{i}-MagicTelescope' for i in range(3)],
    'phoneme': [f'why/classif-num-medium-{i}-phoneme' for i in range(5)],
    'credit': [f'why/classif-num-medium-{i}-credit' for i in range(2)],
    'otto': ['otto'],
}
# TUNING datasets: other benchmark datasets (split 0), used for checks and calibration only
TUNING = {
    'elevators': ['why/regression-num-medium-0-elevators'],
    'house_sales': ['why/regression-cat-medium-0-house_sales'],
    'wine_quality': ['why/regression-num-medium-0-wine_quality'],
    'MiamiHousing2016': ['why/regression-num-medium-0-MiamiHousing2016'],
    'pol': ['why/regression-num-medium-0-pol'],
    'bank-marketing': ['why/classif-num-medium-0-bank-marketing'],
    'kdd_ipums_la_97-small': ['why/classif-num-medium-0-kdd_ipums_la_97-small'],
    'KDDCup09_upselling': ['why/classif-cat-medium-0-KDDCup09_upselling'],
    'jannis': ['why/classif-num-large-0-jannis'],
}
ALL_SPLITS = {**DATASETS, **TUNING}
VARIANTS = ['tabm', 'tabm-mini-piecewiselinear']
missing = [d for ds in ALL_SPLITS.values() for d in ds
           if not os.path.exists(f'{PAPER}/data/' + d.removeprefix('why/') + '/info.json')]
print('All splits present' if not missing else f'MISSING: {missing}')

All splits present


## Cell 3: frozen self-sizing script (`bin/selfsize.py`), copied unchanged from the earlier notebook

In [ ]:
SRC_DIR = '/content/selfsize_src'
os.makedirs(SRC_DIR, exist_ok=True)
open(f'{SRC_DIR}/selfsize_block.py', 'w').write(r'''
# =====================================================================================
# >>> Self-sizing TabM
#
# One training run that shrinks TabM to a FLOPs budget while it trains:
# - width: one L0 hard-concrete gate per neuron (Louizos et al., 2018), shared by all
#   submodels and all blocks, so a closed gate deletes a whole neuron;
# - depth: one L0 gate per block after the first: x <- x + z * (block(x) - x);
#   z = 1 is the normal TabM block, z = 0 is a pass-through, so the block can be deleted;
# - k: greedy submodel selection on the validation set (the TabM[G] rule), done during
#   training;
# - the budget: a MorphNet-style expected-FLOPs term with a self-adjusting multiplier
#   (Lagrangian dual ascent), and a target lowered linearly from 1.0x to the budget.
# Dead neurons, blocks and submodels are physically removed, together with their Adam
# state, so compute really drops. With [selfsize] enabled = false the script is
# identical to bin/model.py (plus cost accounting).
# =====================================================================================

_HC_GAMMA, _HC_ZETA, _HC_BETA = -0.1, 1.1, 2 / 3


class HardConcreteGate(nn.Module):
    def __init__(self, n: int, init: float) -> None:
        super().__init__()
        self.log_alpha = nn.Parameter(torch.full((n,), init) + 0.01 * torch.randn(n))
        # 'off' -> all ones (warm-up), 'sample' -> stochastic (shrink), 'fixed' -> deterministic
        self.mode = 'off'

    def deterministic(self) -> Tensor:
        return (
            torch.sigmoid(self.log_alpha) * (_HC_ZETA - _HC_GAMMA) + _HC_GAMMA
        ).clamp(0, 1)

    def forward(self) -> Tensor:
        if self.mode == 'off':
            return torch.ones_like(self.log_alpha)
        if self.mode == 'fixed' or not self.training:
            return self.deterministic()
        u = torch.rand_like(self.log_alpha).clamp(1e-6, 1 - 1e-6)
        s = torch.sigmoid((u.log() - (1 - u).log() + self.log_alpha) / _HC_BETA)
        return (s * (_HC_ZETA - _HC_GAMMA) + _HC_GAMMA).clamp(0, 1)

    def prob_open(self) -> Tensor:
        return torch.sigmoid(
            self.log_alpha - _HC_BETA * math.log(-_HC_GAMMA / _HC_ZETA)
        )


class GatedMLP(nn.Module):
    """lib.deep.MLP with a shared width gate and per-block depth gates."""

    def __init__(self, mlp: lib.deep.MLP, gate_init: float) -> None:
        super().__init__()
        assert mlp.output is None
        self.blocks = mlp.blocks
        self.output = None
        self.ckpt = False       # True: recompute the blocks in the backward pass (saves memory)
        self.wgate = HardConcreteGate(self.width, gate_init)
        self.dgate = (
            HardConcreteGate(len(self.blocks) - 1, gate_init)
            if len(self.blocks) > 1
            else None
        )

    @property
    def width(self) -> int:
        return self.blocks[0][0].weight.shape[0]

    def gates(self) -> list[HardConcreteGate]:
        return [g for g in (self.wgate, self.dgate) if g is not None]

    def _run(self, x: Tensor, zw: Tensor, zd: None | Tensor) -> Tensor:
        for i, block in enumerate(self.blocks):
            h = block(x) * zw
            x = h if (i == 0 or zd is None) else x + zd[i - 1] * (h - x)
        return x

    def forward(self, x: Tensor) -> Tensor:
        # gates are sampled here, outside the checkpointed part, so the recomputation sees the same gates
        zw = self.wgate()
        zd = None if self.dgate is None else self.dgate()
        if self.ckpt and self.training and torch.is_grad_enabled():
            return torch.utils.checkpoint.checkpoint(self._run, x, zw, zd, use_reentrant=False)
        return self._run(x, zw, zd)


def _ss_flops(d_in: int, d_out: int, k: int, depth: int, width: int) -> float:
    """Forward FLOPs per object of the backbone + heads (multiply-adds)."""
    return k * (d_in * width + (depth - 1) * width * width + width * d_out)


class SelfSizer:
    def __init__(self, model: 'Model', cfg: None | dict, d_out: int, default_cfg: dict) -> None:
        self.model = model
        self.cfg = dict(cfg or {})
        self.enabled = bool(self.cfg.get('enabled', False))
        self.d_in = model.backbone.blocks[0][0].weight.shape[1]
        self.d_out = d_out
        self.flops_default = _ss_flops(
            self.d_in, d_out, default_cfg['k'], default_cfg['n_blocks'], default_cfg['d_block']
        )
        self.lam = 0.0
        self.train_flops = 0.0
        self.trace: list[dict] = []
        self.k_pruned = False
        if not self.enabled:
            return
        c = self.cfg
        self.budget = float(c['budget'])
        self.warmup = int(c.get('warmup_epochs', 3))
        self.target_end = int(c.get('target_end_epoch', 10))
        self.k_prune_epoch = int(c.get('k_prune_epoch', c.get('freeze_epoch', 15)))  # default: once, at the end of shrinking
        self.prune_every = int(c.get('prune_every', 3))
        self.freeze_epoch = int(c.get('freeze_epoch', 15))
        self.dual_lr = float(c.get('dual_lr', 0.03))
        self.gate_lr = float(c.get('gate_lr', 0.05))
        self.k_min = int(c.get('k_min', 4))
        self.k_keep_frac = float(c.get('k_keep_frac', 0.25))
        self.k_tol = float(c.get('k_tol', 0.002))
        self.w_min = int(c.get('w_min', 16))
        # True: k-pruning may not push the model below the budget (the budget is a target);
        # False: k-pruning is purely accuracy-driven, so the budget is only a cap.
        self.k_respect_budget = bool(c.get('k_respect_budget', False))
        # True: physical pruning never takes the model below the current target (budget guard)
        self.budget_guard = bool(c.get('budget_guard', False))
        self.guard_slack = float(c.get('guard_slack', 1.0))
        model.backbone = GatedMLP(model.backbone, float(c.get('gate_init', 2.5)))
        model.backbone.ckpt = bool(c.get('checkpoint', False))
        self.start_ratio = self.actual_flops() / self.flops_default

    # ---------------------------------------------------------------- size and cost
    @property
    def k(self) -> int:
        return self.model.k

    def config(self) -> tuple[int, int, int]:
        bb = self.model.backbone
        return (self.k, len(bb.blocks), bb.blocks[0][0].weight.shape[0])

    def actual_flops(self) -> float:
        return _ss_flops(self.d_in, self.d_out, *self.config())

    def expected_flops(self) -> Tensor:
        bb = self.model.backbone
        w = bb.wgate.prob_open().sum()
        n_hidden = bb.dgate.prob_open().sum() if bb.dgate is not None else 0.0
        return self.k * (self.d_in * w + n_hidden * w * w + w * self.d_out)

    def target(self, epoch: int) -> float:
        if epoch >= self.target_end:
            return self.budget
        t = (epoch - self.warmup) / max(self.target_end - self.warmup, 1)
        return self.start_ratio + (self.budget - self.start_ratio) * min(max(t, 0.0), 1.0)

    # ---------------------------------------------------------------- phases
    def shrinking(self, epoch: int) -> bool:
        return self.enabled and self.warmup < epoch <= self.freeze_epoch

    def size_fixed(self, epoch: int) -> bool:
        """Best-model tracking and early stopping only once the size can no longer change."""
        return (not self.enabled) or epoch > self.freeze_epoch

    def custom_groups(self) -> list[dict]:
        if not self.enabled:
            return []
        params = [p for g in self.model.backbone.gates() for p in g.parameters()]
        return [{'params': params, 'lr': 0.0, 'weight_decay': 0.0, 'selfsize_gates': True}]

    def start_epoch(self, epoch: int, optimizer: torch.optim.Optimizer) -> None:
        if not self.enabled:
            return
        on = self.shrinking(epoch)
        for g in self.model.backbone.gates():
            g.mode = 'sample' if on else ('fixed' if epoch > self.freeze_epoch else 'off')
        for group in optimizer.param_groups:
            if group.get('selfsize_gates'):
                group['lr'] = self.gate_lr if on else 0.0
        self._epoch = epoch

    def add_penalty(self, loss: Tensor) -> Tensor:
        if not (self.enabled and self.shrinking(self._epoch)):
            return loss
        gap = self.expected_flops() / self.flops_default - self.target(self._epoch)
        self._gap = gap.item()
        return loss + self.lam * gap

    def after_step(self, n_objects: int) -> None:
        # n_objects = objects seen by EACH submodel in this step; forward + backward ~ 3x forward
        self.train_flops += 3 * n_objects * self.actual_flops()
        if self.enabled and self.shrinking(self._epoch):
            self.lam = max(0.0, self.lam + self.dual_lr * self._gap)

    def end_epoch(self, epoch: int, optimizer, head_val: np.ndarray, y_val: np.ndarray,
                  task_type: str, val_score: float) -> None:
        if self.enabled:
            # width/depth first, then k, so a budget-aware k step sees the final width and depth
            if self.warmup < epoch <= self.freeze_epoch and (
                (epoch - self.warmup) % self.prune_every == 0 or epoch == self.freeze_epoch
            ):
                guard = self.target(epoch) * self.guard_slack if self.budget_guard else None
                self._prune_width(optimizer, guard)
                self._prune_depth(optimizer, guard)
            if epoch == self.k_prune_epoch and not self.k_pruned:
                if len(self.model.backbone.blocks) and head_val.shape[1] == self.k:
                    self._prune_k(optimizer, self._greedy_k(head_val, y_val, task_type))
                self.k_pruned = True
            if epoch == self.freeze_epoch:
                for g in self.model.backbone.gates():
                    g.mode = 'fixed'
                    g.log_alpha.requires_grad_(False)
        k, depth, width = self.config()
        self.trace.append({
            'epoch': epoch, 'k': k, 'depth': depth, 'width': width,
            'flops_x_default': self.actual_flops() / self.flops_default,
            'expected_flops_x_default': (
                float(self.expected_flops().item()) / self.flops_default if self.enabled else None
            ),
            'target': self.target(epoch) if self.enabled else None,
            'lambda': self.lam, 'val_score': val_score,
        })

    # ---------------------------------------------------------------- pruning
    @staticmethod
    def _replace(module, name, fn, optimizer) -> None:
        old = getattr(module, name)
        new = nn.Parameter(fn(old.data).contiguous(), requires_grad=old.requires_grad)
        setattr(module, name, new)
        st = optimizer.state.pop(old, None)
        if st is not None:
            optimizer.state[new] = {
                kk: (fn(v).contiguous() if torch.is_tensor(v) and v.shape == old.shape else v)
                for kk, v in st.items()
            }
        for g in optimizer.param_groups:
            g['params'] = [new if p is old else p for p in g['params']]

    def _sync_optimizer(self, optimizer) -> None:
        alive = {id(p) for p in self.model.parameters()}
        for g in optimizer.param_groups:
            g['params'] = [p for p in g['params'] if id(p) in alive]
        for p in list(optimizer.state):
            if id(p) not in alive:
                del optimizer.state[p]

    @torch.no_grad()
    def _prune_width(self, optimizer, floor_ratio: None | float = None) -> None:
        bb = self.model.backbone
        z = bb.wgate.deterministic()
        keep = torch.where(z > 0)[0]
        if len(keep) < self.w_min:
            keep = z.topk(self.w_min).indices.sort().values
        if floor_ratio is not None:
            # budget guard: never keep fewer neurons than the current target needs
            k, depth = self.model.k, len(bb.blocks)
            need = next(
                (w for w in range(1, bb.width + 1)
                 if _ss_flops(self.d_in, self.d_out, k, depth, w) >= floor_ratio * self.flops_default),
                bb.width,
            )
            if len(keep) < need:
                keep = z.topk(need).indices.sort().values
        if len(keep) == bb.width:
            return
        sel = lambda dim: (lambda t: t.index_select(dim, keep))
        for i, block in enumerate(bb.blocks):
            lin = block[0]
            self._replace(lin, 'weight', sel(0), optimizer)              # output neurons
            if i > 0:
                self._replace(lin, 'weight', sel(1), optimizer)          # input neurons
                if getattr(lin, 'r', None) is not None:
                    self._replace(lin, 'r', sel(1), optimizer)
            if getattr(lin, 's', None) is not None:
                self._replace(lin, 's', sel(1), optimizer)
            if lin.bias is not None:
                self._replace(lin, 'bias', sel(lin.bias.ndim - 1), optimizer)
            if hasattr(lin, 'out_features'):
                lin.out_features = len(keep)
                if i > 0:
                    lin.in_features = len(keep)
        self._replace(self.model.output, 'weight', sel(1), optimizer)
        self._replace(bb.wgate, 'log_alpha', sel(0), optimizer)

    @torch.no_grad()
    def _prune_depth(self, optimizer, floor_ratio: None | float = None) -> None:
        bb = self.model.backbone
        if bb.dgate is None:
            return
        keep = torch.where(bb.dgate.deterministic() > 0)[0]
        if len(keep) == len(bb.blocks) - 1:
            return
        if floor_ratio is not None and _ss_flops(
                self.d_in, self.d_out, self.model.k, 1 + len(keep), bb.width) < floor_ratio * self.flops_default:
            return                                   # budget guard: this would go below the target
        bb.blocks = nn.ModuleList([bb.blocks[0]] + [bb.blocks[i + 1] for i in keep.tolist()])
        if len(keep) == 0:
            bb.dgate = None
        else:
            self._replace(bb.dgate, 'log_alpha', lambda t: t.index_select(0, keep), optimizer)
        self._sync_optimizer(optimizer)

    @torch.no_grad()
    def _prune_k(self, optimizer, keep: list[int]) -> None:
        if len(keep) == self.k:
            return
        idx = torch.as_tensor(sorted(keep), device=self.model.output.weight.device)
        sel = lambda t: t.index_select(0, idx)
        if self.model.minimal_ensemble_adapter is not None:              # TabM-mini
            self._replace(self.model.minimal_ensemble_adapter, 'weight', sel, optimizer)
        for block in self.model.backbone.blocks:                         # TabM
            lin = block[0]
            for nm in ('r', 's'):
                if getattr(lin, nm, None) is not None:
                    self._replace(lin, nm, sel, optimizer)
            if lin.bias is not None and lin.bias.ndim == 2:
                self._replace(lin, 'bias', sel, optimizer)
            if hasattr(lin, 'k'):
                lin.k = len(idx)
        self._replace(self.model.output, 'weight', sel, optimizer)
        self._replace(self.model.output, 'bias', sel, optimizer)
        self.model.k = len(idx)

    def _greedy_k(self, head_val: np.ndarray, y_val: np.ndarray, task_type: str) -> list[int]:
        """TabM[G]: greedily add the submodel that most improves the COLLECTIVE validation
        loss; keep the smallest subset within k_tol of the full ensemble's loss."""
        P = torch.as_tensor(head_val, dtype=torch.float64)          # (N, k) or (N, k, C)
        y = torch.as_tensor(y_val)
        if task_type == 'regression':
            loss = lambda p: ((p - y.double()[:, None]) ** 2).mean(0)
        elif task_type == 'binclass':
            yy = y.double()[:, None]
            loss = lambda p: -(yy * p.clamp(1e-9, 1).log()
                               + (1 - yy) * (1 - p).clamp(1e-9, 1).log()).mean(0)
        else:
            yi = y.long()
            loss = lambda p: -p.gather(-1, yi.view(-1, 1, 1).expand(-1, p.shape[1], 1)
                                       ).squeeze(-1).clamp_min(1e-9).log().mean(0)
        k = P.shape[1]
        full = loss(P.mean(1, keepdim=True))[0].item()
        chosen, remaining, run_sum, curve = [], list(range(k)), None, []
        for n in range(k):
            cand = P[:, remaining] if run_sum is None else (run_sum.unsqueeze(1) + P[:, remaining]) / (n + 1)
            ls = loss(cand)
            j = int(ls.argmin())
            curve.append(ls[j].item())
            chosen.append(remaining.pop(j))
            run_sum = P[:, chosen[-1]] if run_sum is None else run_sum + P[:, chosen[-1]]
        ok = [i + 1 for i, l in enumerate(curve) if l <= full * (1 + self.k_tol)]
        m = ok[0] if ok else k
        m = max(m, math.ceil(k * self.k_keep_frac), self.k_min)
        if self.k_respect_budget:
            ratio = self.actual_flops() / self.flops_default      # flops scale linearly with k
            m = max(m, math.ceil(k * self.budget / ratio - 1e-9)) if ratio > self.budget else k
        return chosen[: min(m, k)]

    def summary(self) -> dict:
        k, depth, width = self.config()
        return {
            'enabled': self.enabled, 'final_k': k, 'final_depth': depth, 'final_width': width,
            'final_flops_x_default': self.actual_flops() / self.flops_default,
            'train_flops': self.train_flops, 'flops_default': self.flops_default,
            'trace': self.trace,
        }

# <<< Self-sizing TabM
# =====================================================================================

''')
open(f'{SRC_DIR}/make_selfsize.py', 'w').write(r'''"""Create paper/bin/selfsize.py from paper/bin/model.py: same script + self-sizing hooks."""
import sys
from pathlib import Path
paper = Path(sys.argv[1])
src = (paper / 'bin/model.py').read_text()
block = (Path(__file__).parent / 'selfsize_block.py').read_text()

def sub(old, new, count=1):
    global src
    assert src.count(old) == count, (src.count(old), old)
    src = src.replace(old, new)

# 1. the self-sizing classes, before the Config type
sub('class Config(TypedDict):', block + '\n\nclass Config(TypedDict):')
sub('    parameter_statistics: NotRequired[bool]\n',
    '    parameter_statistics: NotRequired[bool]\n    selfsize: NotRequired[KWArgs]\n')
# 2. wrap the model right after it is built
sub("""    report['n_parameters'] = lib.deep.get_n_parameters(model)""",
"""    _defaults = config.get('selfsize', {}).get('default_size', {'k': 32, 'n_blocks': 3, 'd_block': 512})
    sizer = SelfSizer(
        model, config.get('selfsize'),
        d_out=1 if dataset.task.is_regression else dataset.task.compute_n_classes(),
        default_cfg=_defaults,
    )
    report['n_parameters'] = lib.deep.get_n_parameters(model)""")
# 3. gates get their own optimizer group
sub("""**config['optimizer'], params=lib.deep.make_parameter_groups(model)""",
    """**config['optimizer'],
        params=lib.deep.make_parameter_groups(model, custom_groups=sizer.custom_groups())""")
# 4. per-submodel batches follow the CURRENT k
sub("""                    (config['model']['k'], dataset.size('train')),""",
    """                    (model.k, dataset.size('train')),""")
# 5. epoch start, penalty, dual step
sub("""        model.train()
        epoch_losses = []""",
"""        epoch = step // epoch_size + 1
        sizer.start_epoch(epoch, optimizer)
        model.train()
        epoch_losses = []""")
sub("""            loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])""",
"""            loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
            loss = sizer.add_penalty(loss)""")
sub("""            step += 1
            epoch_losses.append(loss.detach())""",
"""            step += 1
            epoch_losses.append(loss.detach())
            sizer.after_step(len(batch_idx) if share_training_batches else len(batch_idx) // model.k)""")
# 6. epoch end: pruning on validation head predictions; no best-model tracking while the size can change
sub("""        metrics, predictions, _, eval_batch_size = evaluate(
            ['val', 'test'], eval_batch_size
        )""",
"""        metrics, predictions, _head_preds, eval_batch_size = evaluate(
            ['val', 'test'], eval_batch_size
        )
        _y_val = dataset.data['y']['val'].cpu().numpy()
        if regression_label_stats is not None:
            _y_val = _y_val * regression_label_stats.std + regression_label_stats.mean
        sizer.end_epoch(
            epoch, optimizer, _head_preds['val'], _y_val,
            dataset.task.type_.value, metrics['val']['score'],
        )""")
sub("""        if (
            'metrics' not in report
            or metrics['val']['score'] > report['metrics']['val']['score']
        ):""",
"""        if not sizer.size_fixed(epoch):
            print()
            continue
        if (
            'metrics' not in report
            or metrics['val']['score'] > report['metrics']['val']['score']
        ):""")
# 7. cost report
sub("""    report['time'] = str(timer)
""",
"""    report['time'] = str(timer)
    report['epochs_trained'] = step // epoch_size
    report['peak_memory_mb'] = (
        torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else None
    )
    report['selfsize'] = sizer.summary()
    report['n_parameters_final'] = lib.deep.get_n_parameters(model)
""")
(paper / 'bin/selfsize.py').write_text(src)
print('wrote', paper / 'bin/selfsize.py')
''')
subprocess.run([sys.executable, f'{SRC_DIR}/make_selfsize.py', PAPER], check=True)
print('bin/selfsize.py ready:', os.path.exists(f'{PAPER}/bin/selfsize.py'))

bin/selfsize.py ready: True


## Cell 4: runner and settings (copied unchanged)

In [ ]:
import json, time, copy, tomli, tomli_w
import numpy as np, pandas as pd
from pathlib import Path

# Self-sizing settings (the ones Step 6 calibrates). Budget = final forward FLOPs as a multiple
# of default TabM's (same variant, same dataset). Schedule in epochs.
SS = dict(
    budget=0.5,
    warmup_epochs=3,        # plain training, gates off
    target_end_epoch=10,    # target lowered linearly from 1.0x to the budget by this epoch
    prune_every=3,          # physically delete closed neurons/blocks every N epochs
    freeze_epoch=15,        # last shrinking epoch; k is pruned here too, then the size is frozen
    dual_lr=0.03,           # how fast the lambda multiplier adapts
    gate_lr=0.05, gate_init=2.5,
    k_min=4, k_keep_frac=0.25, k_tol=0.002, w_min=16,
    k_respect_budget=False, # True: k-pruning may not push the model below the budget
)

def make_config(variant, split, seed, ss=None):
    """Package-default hyperparameters (no per-dataset tuning) on the paper's protocol."""
    cfg = tomli.loads((Path('exp') / variant / split / '0-evaluation' / f'{seed}.toml').read_text())
    has_emb = 'num_embeddings' in cfg['model']
    cfg['amp'] = False
    cfg['head_selection'] = False
    cfg['model']['backbone'].update(n_blocks=2 if has_emb else 3, d_block=512, dropout=0.1)
    cfg['optimizer'].update(lr=2e-3, weight_decay=3e-4)
    if has_emb:
        cfg['model']['num_embeddings']['d_embedding'] = 16
        cfg['bins']['n_bins'] = 48
    default_size = {'k': 32, 'n_blocks': 2 if has_emb else 3, 'd_block': 512}
    cfg['selfsize'] = ({'enabled': False, 'default_size': default_size} if ss is None
                       else {'enabled': True, **ss, 'default_size': default_size})
    return cfg

def train(variant, split, seed, tag, ss=None, overrides=None):
    out = Path('exp/runs') / tag / variant / split
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    cfg = make_config(variant, split, seed, ss)
    for k, v in (overrides or {}).items():
        cfg[k] = v
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, 'bin/selfsize.py', str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag} {variant} {split} {seed}')
    rep = json.loads(report.read_text()); s = rep['selfsize']
    print(f'  {tag:<12} {variant:<26} {split:<44} seed={seed} score={rep["metrics"]["test"]["score"]:.4f} '
          f'size=({s["final_k"]},{s["final_depth"]},{s["final_width"]}) {time.time() - t0:.0f}s')
    return rep

def run(names, tag, seeds, ss=None, table=None, overrides=None):
    table = table or ALL_SPLITS
    for name in names:
        for split in table[name]:
            for variant in VARIANTS:
                for s in seeds:
                    train(variant, split, s, tag, ss, overrides)

def secs(t):
    h, m, s = t.split(':'); return int(h) * 3600 + int(m) * 60 + float(s)

def collect(tag, names, table=None):
    table = table or ALL_SPLITS
    rows = []
    for name in names:
        for split in table[name]:
            for variant in VARIANTS:
                for rep in (Path('exp/runs') / tag / variant / split).glob('*/report.json'):
                    if not rep.parent.name.isdigit():
                        continue
                    r = json.loads(rep.read_text()); s = r['selfsize']
                    rows.append({'dataset': name, 'split': split, 'variant': variant, 'seed': r['config']['seed'],
                                 'score': r['metrics']['test']['score'], 'time_s': secs(r['time']),
                                 'peak_mem_mb': r.get('peak_memory_mb'), 'epochs': r['epochs_trained'],
                                 'train_flops': s['train_flops'], 'final_flops_x': s['final_flops_x_default'],
                                 'size': (s['final_k'], s['final_depth'], s['final_width'])})
    return pd.DataFrame(rows)

def compare(tag, names, base='off', table=None):
    """Self-sizing (tag) vs the same model with self-sizing off (base), per dataset and variant."""
    a, b = collect(base, names, table), collect(tag, names, table)
    out = []
    for (name, variant), g in b.groupby(['dataset', 'variant']):
        h = a[(a.dataset == name) & (a.variant == variant)]
        if h.empty: continue
        sign = -1 if g.score.mean() < 0 else 1                      # RMSE is stored as -RMSE
        out.append({'dataset': name, 'variant': variant, 'runs': len(g),
                    'base_score': round(sign * h.score.mean(), 4), 'base_std': round(h.score.std(), 4),
                    'ss_score': round(sign * g.score.mean(), 4), 'ss_std': round(g.score.std(), 4),
                    'rel_change_%': round((g.score.mean() - h.score.mean()) / abs(h.score.mean()) * 100, 2),
                    'final_flops_x': round(g.final_flops_x.mean(), 3),
                    'train_flops_x': round(g.train_flops.mean() / h.train_flops.mean(), 2),
                    'time_x': round(g.time_s.mean() / h.time_s.mean(), 2),
                    'mem_x': (round(g.peak_mem_mb.mean() / h.peak_mem_mb.mean(), 2)
                              if g.peak_mem_mb.notna().all() and h.peak_mem_mb.notna().all() else None),
                    'final_sizes': sorted(set(g['size']))})
    df = pd.DataFrame(out)
    df.to_csv(f'{ROOT}/compare_{tag}.csv', index=False)
    return df

In [ ]:
import json, time, subprocess, sys
import numpy as np, pandas as pd, tomli_w
from pathlib import Path

FROZEN_SS = {**SS, 'budget': 0.5, 'budget_guard': True, 'k_keep_frac': 1.0,
             'k_respect_budget': False, 'guard_slack': 0.8}
GENTLE_SS = {**FROZEN_SS, 'warmup_epochs': 6, 'target_end_epoch': 20, 'freeze_epoch': 25}
K = 32
SEEDS = [0, 1, 2]
ITEMS = [('cpu_act', 'why/regression-num-medium-0-cpu_act'),
         ('superconduct', 'why/regression-num-medium-0-superconduct')]

def cfg_with_k(variant, split, seed, k, ss=None):
    cfg = make_config(variant, split, seed, ss)
    cfg['model']['k'] = int(k)
    return cfg

def plain_cfg(split, seed, width):
    cfg = cfg_with_k('tabm', split, seed, K, None)
    cfg['model']['backbone']['d_block'] = int(width)
    return cfg

def run_cfg(cfg, out, seed, tag):
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, 'bin/selfsize.py', str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag}')
    rep = json.loads(report.read_text())
    print(f'  {tag:<36} score={rep["metrics"]["test"]["score"]:.4f} '
          f'size=({rep["selfsize"]["final_k"]},{rep["selfsize"]["final_depth"]},{rep["selfsize"]["final_width"]}) {time.time()-t0:.0f}s')
    return rep

def sc(r): return r['metrics']['test']['score']
print('settings ready')

settings ready


## Cell 5: distillation patch -> `bin/selfsize_distill.py`
Generated from `bin/selfsize.py` by three insertions (config key, target blend, report field). No existing line is changed.
The cell prints the diff so you can check that.

In [ ]:
open(f'{SRC_DIR}/make_distill.py', 'w').write(r'''"""Create paper/bin/selfsize_distill.py from paper/bin/selfsize.py (the frozen self-sizing script).
selfsize.py itself is never modified, so every earlier (frozen/default/plain) run is unaffected.

Distillation for regression, implemented as a target blend:
    (1-l)*(p-y)^2 + l*(p-t)^2  =  (p - m)^2 + l*(1-l)*(y-t)^2,   m = (1-l)*y + l*t
The last term does not depend on the prediction p, so the gradient of the two-term loss equals the
gradient of ONE MSE against the blended target m. So the only change is Y_train := m, computed once.
Teacher t = the default TabM ensemble-mean prediction on the TRAIN rows (same seed, same split)."""
import sys
from pathlib import Path
paper = Path(sys.argv[1])
src = (paper / 'bin/selfsize.py').read_text()

def sub(old, new):
    global src
    assert src.count(old) == 1, (src.count(old), old)
    src = src.replace(old, new)

sub('    selfsize: NotRequired[KWArgs]\n',
    '    selfsize: NotRequired[KWArgs]\n    distill: NotRequired[KWArgs]\n')

sub('    # >>> Training\n    step = 0\n', """    # >>> Distillation (regression only): Y_train := (1-l)*y + l*teacher, see make_distill.py
    _dcfg = config.get('distill') or {}
    distill_info = {'enabled': False}
    if _dcfg.get('enabled', False):
        assert dataset.task.is_regression, 'distillation is implemented for regression only'
        _lam = float(_dcfg['lambda'])
        assert 0.0 <= _lam <= 1.0, _lam
        _t = np.load(_dcfg['teacher_predictions'])['train'].astype('float64')
        assert _t.shape == (dataset.size('train'),), (_t.shape, dataset.size('train'))
        assert np.isfinite(_t).all(), 'teacher predictions contain non-finite values'
        _t = (_t - regression_label_stats.mean) / regression_label_stats.std     # same scale as Y_train
        T_train = torch.as_tensor(_t, dtype=torch.float, device=Y_train.device)
        _gap = float(((T_train - Y_train) ** 2).mean().sqrt())                    # label std = 1
        distill_info = {
            'enabled': True, 'lambda': _lam, 'teacher_predictions': str(_dcfg['teacher_predictions']),
            'teacher_train_rmse_in_label_std': _gap,
        }
        logger.info(f'distillation: lambda={_lam}, teacher-vs-label train RMSE = {_gap:.4f} label stds')
        Y_train = (1.0 - _lam) * Y_train + _lam * T_train
    # <<< Distillation

    # >>> Training
    step = 0
""")

sub("    report['selfsize'] = sizer.summary()\n",
    "    report['selfsize'] = sizer.summary()\n    report['distill'] = distill_info\n")

(paper / 'bin/selfsize_distill.py').write_text(src)
print('wrote', paper / 'bin/selfsize_distill.py')
''')
subprocess.run([sys.executable, f'{SRC_DIR}/make_distill.py', PAPER], check=True)
subprocess.run([sys.executable, '-m', 'py_compile', f'{PAPER}/bin/selfsize_distill.py'], check=True)
d = subprocess.run(['diff', f'{PAPER}/bin/selfsize.py', f'{PAPER}/bin/selfsize_distill.py'], capture_output=True, text=True).stdout
print(d)
assert not any(l.startswith('<') for l in d.splitlines()), 'patch must only ADD lines'
print('patch ok: only additions')

661a662
>     distill: NotRequired[KWArgs]
790a792,812
>     # >>> Distillation (regression only): Y_train := (1-l)*y + l*teacher, see make_distill.py
>     _dcfg = config.get('distill') or {}
>     distill_info = {'enabled': False}
>     if _dcfg.get('enabled', False):
>         assert dataset.task.is_regression, 'distillation is implemented for regression only'
>         _lam = float(_dcfg['lambda'])
>         assert 0.0 <= _lam <= 1.0, _lam
>         _t = np.load(_dcfg['teacher_predictions'])['train'].astype('float64')
>         assert _t.shape == (dataset.size('train'),), (_t.shape, dataset.size('train'))
>         assert np.isfinite(_t).all(), 'teacher predictions contain non-finite values'
>         _t = (_t - regression_label_stats.mean) / regression_label_stats.std     # same scale as Y_train
>         T_train = torch.as_tensor(_t, dtype=torch.float, device=Y_train.device)
>         _gap = float(((T_train - Y_train) ** 2).mean().sqrt())                    # label std = 1
>     

## Cell 6: helpers (runner for either script, teacher lookup, configs)

In [ ]:
DISTILL_LAMBDAS = {'distill-l050': 0.5, 'distill-l100': 1.0}     # pre-declared, not tuned
ITEMS = [('cpu_act', 'why/regression-num-medium-0-cpu_act'),
         ('superconduct', 'why/regression-num-medium-0-superconduct')]
SEEDS = [0, 1, 2]
FREEZE = FROZEN_SS['freeze_epoch']

def run_cfg2(cfg, out, seed, tag, script='bin/selfsize.py'):
    """Like run_cfg, but selects the script. Resumable: an existing report.json is reused."""
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, script, str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag}')
    rep = json.loads(report.read_text())
    print(f'  {tag:<40} score={sc(rep):.4f} size=({rep["selfsize"]["final_k"]},{rep["selfsize"]["final_depth"]},'
          f'{rep["selfsize"]["final_width"]}) epochs={rep["epochs_trained"]} {time.time()-t0:.0f}s')
    return rep

def run_dir(tag, split): return Path('exp/runs') / tag / 'tabm' / split

def ensure_teacher(split, seed):
    """Default TabM run for (split, seed). Its predictions.npz already holds the ensemble-mean TRAIN predictions
    (the script evaluates ['train','val','test'] after loading the best checkpoint). Reused if present."""
    out = run_dir('default', split)
    pred = out / str(seed) / 'predictions.npz'
    rep_path = out / str(seed) / 'report.json'
    if rep_path.exists() and not (pred.exists() and 'train' in np.load(pred).files):
        print(f'  teacher {split} s{seed}: predictions.npz has no train part, rerunning'); rep_path.unlink()
    rep = run_cfg2(cfg_with_k('tabm', split, seed, K, None), out, seed, f'default {split} s{seed}')
    assert pred.exists() and 'train' in np.load(pred).files, f'no train predictions for {split} s{seed}'
    return rep, pred.resolve()

def distill_cfg(split, seed, lam, pred_path, ss=FROZEN_SS):
    cfg = cfg_with_k('tabm', split, seed, K, ss)
    cfg['distill'] = {'enabled': True, 'lambda': float(lam), 'teacher_predictions': str(pred_path)}
    return cfg
print('helpers ready')

helpers ready


## Cell 7: unit test of the exact-equivalence claim (CPU, no data, seconds)
Checks numerically that the two-term loss and the blended-target MSE have the same gradient.

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)
p = torch.randn(256, 32, requires_grad=True); y = torch.randn(256, 32); t = torch.randn(256, 32)
for lam in [0.0, 0.25, 0.5, 1.0]:
    g_two, = torch.autograd.grad((1 - lam) * F.mse_loss(p, y) + lam * F.mse_loss(p, t), p)
    g_one, = torch.autograd.grad(F.mse_loss(p, (1 - lam) * y + lam * t), p)
    err = (g_two - g_one).abs().max().item()
    assert err < 1e-6, (lam, err)
    print(f'lambda={lam}: max |grad difference| = {err:.2e}')
print('equivalence holds')

lambda=0.0: max |grad difference| = 0.00e+00
lambda=0.25: max |grad difference| = 1.16e-10
lambda=0.5: max |grad difference| = 1.16e-10
lambda=1.0: max |grad difference| = 0.00e+00
equivalence holds


## Cell 8: smoke test on the real cpu_act split (about 1 to 2 minutes)
Runs a very short schedule (4 epochs, shrinking in epochs 2 to 3) so that the distillation path and the pruning path
are both exercised. Checks: the run finishes, scores are finite, the report records `distill.enabled`, and the teacher
array has one value per training row. Also compares `selfsize_distill.py` with distillation OFF against the frozen
`selfsize.py` on the same short schedule; GPU kernels are not bit-deterministic, so a small difference is only warned about.

In [ ]:
split = ITEMS[0][1]; seed = 0
_, pred = ensure_teacher(split, seed)
SMOKE_SS = {**FROZEN_SS, 'warmup_epochs': 1, 'target_end_epoch': 2, 'freeze_epoch': 3, 'prune_every': 1}
ov = {'n_epochs': 4}

def smoke(tag, cfg, script):
    cfg = {**cfg, **ov}
    return run_cfg2(cfg, Path('exp/runs') / 'smoke' / tag / 'tabm' / split, seed, f'smoke {tag}', script)

a = smoke('frozen-short', cfg_with_k('tabm', split, seed, K, SMOKE_SS), 'bin/selfsize.py')
b = smoke('distill-off', cfg_with_k('tabm', split, seed, K, SMOKE_SS), 'bin/selfsize_distill.py')
c = smoke('distill-on', distill_cfg(split, seed, 0.5, pred, SMOKE_SS), 'bin/selfsize_distill.py')

n_train = len(np.load(pred)['train'])
assert c['distill']['enabled'] and not b['distill']['enabled'], 'distill flag not recorded'
assert all(np.isfinite(sc(r)) for r in (a, b, c)), 'non-finite score'
assert all(r['epochs_trained'] >= 4 for r in (a, b, c)), 'a smoke run stopped early'
print('teacher rows:', n_train, '| teacher-vs-label train RMSE (label stds):', round(c['distill']['teacher_train_rmse_in_label_std'], 4))
diff = abs(sc(a) - sc(b))
print(f'frozen vs distill-OFF on the same seed: |score difference| = {diff:.2e}',
      '(ok)' if diff < 1e-3 else '(WARNING: larger than expected; check GPU nondeterminism before trusting comparisons)')
print('smoke test passed')

  default why/regression-num-medium-0-cpu_act s0 score=-2.0681 size=(32,3,512) epochs=121 71s
  smoke frozen-short                       score=-3.0706 size=(32,3,511) epochs=4 16s
  smoke distill-off                        score=-3.0706 size=(32,3,511) epochs=4 16s
  smoke distill-on                         score=-3.0769 size=(32,3,511) epochs=4 16s
teacher rows: 5734 | teacher-vs-label train RMSE (label stds): 0.0742
frozen vs distill-OFF on the same seed: |score difference| = 0.00e+00 (ok)
smoke test passed


## Cell 9: the runs (default = teacher, frozen control, frozen-width plain, two distillation arms). Resumable.

In [ ]:
for n, s in ITEMS:
    for seed in SEEDS:
        d, pred = ensure_teacher(s, seed)
        f = run_cfg2(cfg_with_k('tabm', s, seed, K, FROZEN_SS), run_dir('frozen-self', s), seed, f'frozen self {n} s{seed}')
        run_cfg2(plain_cfg(s, seed, f['selfsize']['final_width']), run_dir('frozen-plain', s), seed,
                 f'frozen plain {n} w{f["selfsize"]["final_width"]} s{seed}')
        for tag, lam in DISTILL_LAMBDAS.items():
            run_cfg2(distill_cfg(s, seed, lam, pred), run_dir(tag, s), seed, f'{tag} {n} s{seed}',
                     'bin/selfsize_distill.py')
print('all runs done')

  frozen self cpu_act s0                   score=-2.2127 size=(32,3,322) epochs=115 51s
  frozen plain cpu_act w322 s0             score=-2.0928 size=(32,3,322) epochs=120 43s
  distill-l050 cpu_act s0                  score=-2.2226 size=(32,3,322) epochs=111 49s
  distill-l100 cpu_act s0                  score=-2.2258 size=(32,3,322) epochs=112 50s
  default why/regression-num-medium-0-cpu_act s1 score=-2.0928 size=(32,3,512) epochs=88 49s
  frozen self cpu_act s1                   score=-2.1753 size=(32,3,322) epochs=115 52s
  frozen plain cpu_act w322 s1             score=-2.0510 size=(32,3,322) epochs=137 49s
  distill-l050 cpu_act s1                  score=-2.2269 size=(32,3,322) epochs=94 46s
  distill-l100 cpu_act s1                  score=-2.2568 size=(32,3,322) epochs=94 45s
  default why/regression-num-medium-0-cpu_act s2 score=-2.0728 size=(32,3,512) epochs=106 55s
  frozen self cpu_act s2                   score=-2.1633 size=(32,3,322) epochs=115 49s
  frozen plain cpu_act 

## Cell 10: summary
Score is stored as -RMSE (higher is better); percentages are relative to the reference score, positive = better.
Validity check: every shrinking run must have trained past `freeze_epoch`; invalid runs are flagged.

In [ ]:
def load(tag, s, seed): return json.loads((run_dir(tag, s) / str(seed) / 'report.json').read_text())
def pct(a, b): return (sc(a) - sc(b)) / abs(sc(b)) * 100

rows, diag = [], []
for n, s in ITEMS:
    for seed in SEEDS:
        d, f, fp = load('default', s, seed), load('frozen-self', s, seed), load('frozen-plain', s, seed)
        diag.append({'dataset': n, 'seed': seed,
                     'teacher_train_RMSE': -d['metrics']['train']['score'], 'teacher_val_RMSE': -d['metrics']['val']['score'],
                     'teacher_test_RMSE': -d['metrics']['test']['score']})
        row = {'dataset': n, 'seed': seed, 'frozen_vs_default_%': pct(f, d), 'frozen_vs_plain_%': pct(f, fp),
               'frozen_width': f['selfsize']['final_width']}
        valid = f['epochs_trained'] > FREEZE
        for tag in DISTILL_LAMBDAS:
            r = load(tag, s, seed)
            valid &= r['epochs_trained'] > FREEZE
            row[f'{tag}_vs_default_%'] = pct(r, d)
            row[f'{tag}_vs_frozen_%'] = pct(r, f)
            row[f'{tag}_vs_frozen_plain_%'] = pct(r, fp)
            row[f'{tag}_width'] = r['selfsize']['final_width']
        row['valid'] = valid
        rows.append(row)
df = pd.DataFrame(rows); dg = pd.DataFrame(diag)
df.to_csv(f'{ROOT}/distill_sample.csv', index=False); dg.to_csv(f'{ROOT}/distill_teacher_diag.csv', index=False)

print('Teacher diagnostic (default TabM, RMSE in label units):')
print(dg.round(4).to_string()); print()
print('Per run (percent change in test RMSE-score; positive = better):')
print(df.round(2).to_string()); print()
cols = [c for c in df.columns if c.endswith('%')]
print('Mean over seeds:'); print(df.groupby('dataset')[cols].mean().round(2).to_string()); print()
print('Std over seeds:');  print(df.groupby('dataset')[cols].std().round(2).to_string()); print()
print('Runs that did NOT pass the epoch check:', int((~df.valid).sum()), '(must be 0)')
print('Read-out rule: with 3 seeds treat differences as observations. A distillation gain counts only if it is larger than the\n'
      'seed std AND has the same sign on all 3 seeds on both datasets. Report as post-hoc, not as a validated improvement.')

Teacher diagnostic (default TabM, RMSE in label units):
        dataset  seed  teacher_train_RMSE  teacher_val_RMSE  teacher_test_RMSE
0       cpu_act     0              1.3539            2.1319             2.0681
1       cpu_act     1              1.5747            2.1640             2.0928
2       cpu_act     2              1.4342            2.1485             2.0728
3  superconduct     0              6.7184            9.8080            10.2666
4  superconduct     1              6.3415            9.6772            10.2587
5  superconduct     2              6.6352            9.7567            10.2487

Per run (percent change in test RMSE-score; positive = better):
        dataset  seed  frozen_vs_default_%  frozen_vs_plain_%  frozen_width  distill-l050_vs_default_%  distill-l050_vs_frozen_%  distill-l050_vs_frozen_plain_%  distill-l050_width  distill-l100_vs_default_%  distill-l100_vs_frozen_%  distill-l100_vs_frozen_plain_%  distill-l100_width  valid
0       cpu_act     0            